## JsonServer

Dexedを起動すると TCP サーバーが自動起動する。  
JSONを送信することでパラメータ変更・WAVレンダリング・プリセット切り替えを外部から制御できる。

デフォルトポートは8765、複数起動時は8781まで使用。

---
## 1. 接続クライアント

In [ ]:
import socket
import json
import time


class DexedClient:
    def __init__(self, host: str = "127.0.0.1", port: int = 8765, timeout: float = 10.0):
        self.host = host
        self.port = port
        self.timeout = timeout
        self._sock: socket.socket | None = None
        self._buf = b""

    def connect(self):
        self._sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self._sock.settimeout(self.timeout)
        self._sock.connect((self.host, self.port))
        self._buf = b""

    def close(self):
        if self._sock:
            self._sock.close()
            self._sock = None

    def __enter__(self):
        self.connect()
        return self

    def __exit__(self, *_):
        self.close()

    def _send(self, obj: dict):
        line = json.dumps(obj, ensure_ascii=False) + "\n"
        self._sock.sendall(line.encode("utf-8"))

    def _recv_line(self) -> dict:
        while b"\n" not in self._buf:
            chunk = self._sock.recv(65536)
            if not chunk:
                raise ConnectionError("Server closed the connection.")
            self._buf += chunk
        line, self._buf = self._buf.split(b"\n", 1)
        return json.loads(line.decode("utf-8"))

    def _send_recv(self, obj: dict) -> dict:
        self._send(obj)
        return self._recv_line()

    def query(self) -> dict:
        return self._send_recv({"query": "all"})

    def set_params(self, params: dict) -> None:
        self._send(params)

    def load_syx(self, path: str, program: int = 0) -> dict:
        return self._send_recv({"load_syx": path, "program": program})

    def set_program(self, program: int) -> dict:
        return self._send_recv({"set_program": program})

    def render(self, output: str, midi_note: int = 60, velocity: float = 0.8) -> dict:
        return self._send_recv({"render": output, "midi_note": midi_note, "velocity": velocity})

    def batch(self, items: list[dict], midi_note: int = 60, velocity: float = 0.8) -> dict:
        return self._send_recv({"batch": items, "midi_note": midi_note, "velocity": velocity})


---
## 2. 接続

複数インスタンスを起動している場合はポート番号を変更する。

In [2]:
PORT = 8765

client = DexedClient(port=PORT)
client.connect()

---
## 3. query — 全パラメータ取得

全パラメータ値が正規化値 (0.0〜1.0) で返る。

In [3]:
params = client.query()

print(f"パラメータ数: {len(params)}")
print()

for name, value in list(params.items())[:10]:
    print(f"  {name:<30} = {value:.4f}")
print("  ...")

パラメータ数: 156

  Cutoff                         = 0.6571
  Resonance                      = 0.2578
  Output                         = 0.8475
  MonoMode                       = 0.0000
  MASTER TUNE ADJ                = 0.7314
  ALGORITHM                      = 0.0000
  FEEDBACK                       = 0.0000
  OSC KEY SYNC                   = 1.0000
  LFO SPEED                      = 0.3535
  LFO DELAY                      = 0.0000
  ...


---
## 4. パラメータ設定

`query` で取得したキー名をそのまま使う。レスポンスはない。

In [ ]:
original = client.query()

algo_key = [k for k in original if "ALGORITHM" in k.upper()]
print(f"Algorithm キー: {algo_key}")

if algo_key:
    key = algo_key[0]
    print(f"変更前: {key} = {original[key]:.4f}")

    client.set_params({key: 2 / 31})
    time.sleep(0.1)

    after = client.query()
    print(f"変更後: {key} = {after[key]:.4f}")
    time.sleep(3)

    client.set_params({key: original[key]})
    print("元の値に戻しました")

Algorithm キー: ['ALGORITHM']
変更前: ALGORITHM = 0.6129
変更後: ALGORITHM = 0.0645
元の値に戻しました


---
## 5. set_program — プログラム切り替え

カートリッジ内のプログラムを切り替える (0〜31)。

In [ ]:
for prog_num in range(4):
    resp = client.set_program(prog_num)
    print(f"  Program {resp['program']:2d}: {resp['program_name']}")

print()
print(json.dumps(resp, ensure_ascii=False, indent=2))

---
## 6. load_syx — SYXファイルのロード

`load_syx` には絶対パス、`program` はロード後に選択するプログラム番号 (0–31)。

In [ ]:
import os

SYX_PATH = r"C:\path\to\your\presets.syx"  # 実際のパスに書き換える

if not os.path.exists(SYX_PATH):
    print(f"ファイルが存在しません: {SYX_PATH}")
else:
    resp = client.load_syx(SYX_PATH, program=0)
    print(json.dumps(resp, ensure_ascii=False, indent=2))

    print("\nカートリッジ内のプログラム一覧:")
    for i in range(32):
        r = client.set_program(i)
        print(f"  [{i:2d}] {r['program_name']}")

---
## 7. render — WAVレンダリング

出力: 48kHz / 32bit float / モノラル / 約4秒 (120BPM × 8拍)

In [5]:
import tempfile
import pathlib

OUT_DIR = pathlib.Path(tempfile.gettempdir()) / "dexed_test"
OUT_DIR.mkdir(exist_ok=True)

out_wav = str(OUT_DIR / "render_test.wav")

resp = client.render(out_wav, midi_note=60, velocity=0.8)
print("レスポンス:", resp)

time.sleep(3.0)

p = pathlib.Path(out_wav)
if p.exists():
    print(f"{p}  ({p.stat().st_size / 1024:.1f} KB)")
else:
    print("ファイルが存在しません")

レスポンス: {'ok': True}
V:\Temp\dexed_test\render_test.wav  (1500.2 KB)


In [ ]:
try:
    import numpy as np
    import scipy.io.wavfile as wav
    import matplotlib.pyplot as plt

    sr, data = wav.read(out_wav)
    t = np.arange(len(data)) / sr

    plt.figure(figsize=(12, 3))
    plt.plot(t, data, linewidth=0.3)
    plt.xlabel("Time [s]")
    plt.ylabel("Amplitude")
    plt.title(f"Dexed render — {sr} Hz, {len(data)} samples")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("pip install scipy matplotlib numpy")

---
## 8. batch — 一括レンダリング

全アイテムのレンダリング完了後にレスポンスが返る。  
各アイテムに `midi_note` / `velocity` を指定するとトップレベルの値を上書きできる。

アイテムのフィールド: `output`（必須）、`set_program`、`load_syx` + `program`、`params`

In [ ]:
batch_items = [
    {"output": str(OUT_DIR / f"batch_{i:02d}.wav"), "set_program": i}
    for i in range(3)
]

start = time.time()
resp = client.batch(batch_items, midi_note=60, velocity=0.8)
print(f"完了: {time.time() - start:.1f} 秒")
print(json.dumps(resp, ensure_ascii=False, indent=2))

print()
for item in batch_items:
    p = pathlib.Path(item["output"])
    status = f"{p.stat().st_size / 1024:.1f} KB" if p.exists() else "NOT FOUND"
    print(f"  {p.name}: {status}")

### 8a. params を使った音色変形バッチ

In [ ]:
current = client.query()

rate_keys = [k for k in current if "EG RATE" in k]
print("EG Rate パラメータ:", rate_keys[:6], "...")

if rate_keys:
    first_rate = rate_keys[0]

    variation_items = [
        {
            "output": str(OUT_DIR / f"variation_{i}.wav"),
            "set_program": 0,
            "params": {first_rate: v},
        }
        for i, v in enumerate([0.2, 0.5, 0.8])
    ]

    print(f"{first_rate} を 0.2 / 0.5 / 0.8 で書き出し...")
    resp = client.batch(variation_items)
    print(json.dumps(resp, ensure_ascii=False, indent=2))

---
## 9. 切断

In [ ]:
client.close()